# Video decomposition test
In this workbook you can decompose video file using the previously generated Gabor library space, similar to Skriabine S. et al 2026. Then visualize results.

In [16]:
from pathlib import Path
import numpy as np
from analysis_utils import downscale_binary_video
from wavelet_utils_vSpeed import loadFilterParamDict_vS, compute_and_save_dwt_vS, makeFilterParamDict_vS, saveFilterParamDict_vS, filename_fromFilterParam, get_filter_from_params, get_filter_vector

In [17]:
full_screen_coverage = [0, 88, -17, 49] # [az_left, az_right, el_bottom, el_top] full screen position in visual degrees
visual_coverage = [0, 88, -17, 49] # [az_left, az_right, el_bottom, el_top] screen coverage in visual degrees (seen by the animal)

screen_x = 100 # horizontal screen size in pixels for the Gabor filter generation and movie analysis
screen_t = 3 # time dimension of the screen in pixels

nx = 16 # number of Gabor filters in the horizontal direction (azimuth) (y will be generated)
        # 40 is a good number. Use 5 for testing to make it fast
        
n_thetas = 8 # number of angles to generate
theta_max = np.pi

size_min = 5 # minimum size in visual degrees
size_max = 25 # maximum size in visual degrees
n_sizes = 5   # number of sizes to generate

freq_min = .025 # minimum frequency in cycles per visual degree
freq_max = .15 # maximum frequency in cycles per visual degree
n_freqs = 5  # number of frequencies to generate

n_phases = 2  # number of phases to generate
phase_max = np.pi*1 # maximum phase to generate

driftmax=2 # maximum drift speed in degrees per frame
driftnum=3 # 2n+1 drift speeds will be generated

In [18]:
#calculations
az_left, az_right, el_bottom, el_top = visual_coverage

screen_y = int(screen_x * (el_top - el_bottom) / (az_right - az_left))
ny = int(nx * (el_top - el_bottom) / (az_right - az_left))

# centers in visual degrees
xs = np.linspace(az_left, az_right, nx, endpoint=False)+(az_right - az_left) / (2*nx)
ys = np.linspace(el_bottom, el_top, ny, endpoint=False)+(el_top - el_bottom) / (2*ny)

angles= np.linspace(0, theta_max, n_thetas, endpoint=False)
sizes = np.logspace(np.log10(size_min), np.log10(size_max), n_sizes)
freqs = np.logspace(np.log10(freq_min), np.log10(freq_max), n_freqs)
phases = np.linspace(0,  phase_max, n_phases, endpoint=False)
drifts=np.linspace(-driftmax, driftmax, driftnum*2+1)

print(f"Screen size: {screen_x}x{screen_y} pixels")
print(f"Full screen coverage: {full_screen_coverage} degrees")
print(f"Visual coverage: {visual_coverage} degrees")
print(f"Center positions (x_deg): {np.round(xs, 1)} degrees")
print(f"Center positions (y_deg): {np.round(ys, 1)} degrees")
print(f"Angles (degrees): {np.round(np.rad2deg(angles), 1)}")
print(f"Sizes (degrees): {sizes}")
print(f"Frequencies (cycles/degree): {freqs}")
print(f"Phases (degrees): {np.rad2deg(phases)}")
print(f"Drifts (degrees/frame): {drifts}")

Screen size: 100x75 pixels
Full screen coverage: [0, 88, -17, 49] degrees
Visual coverage: [0, 88, -17, 49] degrees
Center positions (x_deg): [ 2.8  8.2 13.8 19.2 24.8 30.2 35.8 41.2 46.8 52.2 57.8 63.2 68.8 74.2
 79.8 85.2] degrees
Center positions (y_deg): [-14.2  -8.8  -3.2   2.2   7.8  13.2  18.8  24.2  29.8  35.2  40.8  46.2] degrees
Angles (degrees): [  0.   22.5  45.   67.5  90.  112.5 135.  157.5]
Sizes (degrees): [ 5.          7.47674391 11.18033989 16.71850762 25.        ]
Frequencies (cycles/degree): [0.025      0.03912711 0.06123724 0.09584147 0.15      ]
Phases (degrees): [ 0. 90.]
Drifts (degrees/frame): [-2.         -1.33333333 -0.66666667  0.          0.66666667  1.33333333
  2.        ]


In [19]:
total_n=len(sizes)*len(angles)*len(freqs)*len(drifts)*len(phases)*len(xs)*len(ys)
print(f"Total number of Gabor filters to generate: {total_n}")

Total number of Gabor filters to generate: 537600


In [20]:
# a control
gabor_step=(az_right-az_left)/nx 
print(f"Control: Gabor placement step in visual degrees (x): {gabor_step:.1f}, vs size_min: {size_min:.1f} degrees. {'OK' if (gabor_step < size_min) else 'WARNING!'}")
gabor_step=(el_top-el_bottom)/ny
print(f"Control: Gabor placement step in visual degrees (y): {gabor_step:.1f}, vs size_min: {size_min:.1f} degrees. {'OK' if (gabor_step < size_min) else 'WARNING!'}")
visual_step_x=(az_right-az_left)/screen_x
print(f"Control: Gabor resolution in visual degrees (x): {visual_step_x:.1f}, vs 1/freq_max: {1/freq_max:.1f} degrees. {'OK' if (visual_step_x < 1/freq_max/4) else 'WARNING!'}")
visual_step_y=(el_top-el_bottom)/screen_y
print(f"Control: Gabor resolution in visual degrees (y): {visual_step_y:.1f}, vs 1/freq_max: {1/freq_max:.1f} degrees. {'OK' if (visual_step_y < 1/freq_max/4) else 'WARNING!'}")


Control: Gabor placement step in visual degrees (x): 5.5, vs size_min: 5.0 degrees. WARNING!
Control: Gabor placement step in visual degrees (y): 5.5, vs size_min: 5.0 degrees. WARNING!
Control: Gabor resolution in visual degrees (x): 0.9, vs 1/freq_max: 6.7 degrees. OK
Control: Gabor resolution in visual degrees (y): 0.9, vs 1/freq_max: 6.7 degrees. OK


In [21]:

temppath = r'D:\SynologyDriveSyncedDATA\PROCESSED\Waven'

wavelet_params=makeFilterParamDict_vS(screen_x, screen_y, screen_t, visual_coverage, full_screen_coverage, xs, ys, angles, sizes, freqs, drifts, phases)

_, paramname = filename_fromFilterParam(wavelet_params)
paramspath = Path(temppath) / paramname
saveFilterParamDict_vS(wavelet_params, paramspath)
print(f"Saved Gabor filter parameters to {paramspath}")


videopath = Path(temppath) / 'zebra_s0_d420.0_fps59.94_RESAMPLED30fps.mp4'
#videopath = Path(temppath) / 'zebra_s1_d420.0_fps59.94_RESAMPLED30fps.mp4'

Saved Gabor filter parameters to D:\SynologyDriveSyncedDATA\PROCESSED\Waven\gaborLibrary_vS_16_12_8_5_5_7_2_362db4c1.json


In [22]:

xs, ys, angles, sizes, freqs, drifts, phases, visual_coverage, full_screen_coverage, screen_t, screen_x, screen_y = loadFilterParamDict_vS(paramspath)


In [23]:
print(f"full_screen_coverage: {full_screen_coverage}, visual_coverage: {visual_coverage}")

full_screen_coverage: [  0  88 -17  49], visual_coverage: [  0  88 -17  49]


## Downsample video

In [24]:
downsampled_video_path, optic_flow_path=downscale_binary_video(videopath, full_screen_coverage, visual_coverage, screen_x, screen_y, force=False, generate_optic_flow=True, fps=30)

Generating cropped and downsampled binary video...
Output file D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_scaled100x75.npy already exists. Skipping generation.


## Display downsampled video result

In [25]:
# Displaying original and downsampled video frames
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from pathlib import Path

input_video_path = Path(videopath)
#output_npy_path 

out_video = np.load(downsampled_video_path, mmap_mode="r")

cap = cv2.VideoCapture(str(input_video_path))
n_frames = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), out_video.shape[0])

def show_frame(frame_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret:
        print(f"Could not read frame {frame_idx}")
        return

    input_gray = frame[:, :, 0]
    output_frame = out_video[frame_idx].T

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(
        input_gray,
        cmap="gray",
        extent=[full_screen_coverage[0], full_screen_coverage[1],
                full_screen_coverage[2], full_screen_coverage[3]],
        aspect="equal",
    )
    axes[0].set_title(f"Input frame {frame_idx}")
    axes[0].set_xlabel("Azimuth (°)")
    axes[0].set_ylabel("Elevation (°)")

    axes[1].imshow(
        output_frame,
        cmap="gray",
        vmin=0,
        vmax=1,
        extent=[visual_coverage[0], visual_coverage[1],
                visual_coverage[2], visual_coverage[3]],
        aspect="equal",
        origin="lower"
    )
    axes[1].set_title(f"Output frame {frame_idx}")
    axes[1].set_xlabel("Azimuth (°)")
    axes[1].set_ylabel("Elevation (°)")

    plt.tight_layout()
    plt.show()

interact(
    show_frame,
    frame_idx=IntSlider(
        value=0,
        min=0,
        max=n_frames - 1,
        step=1,
        description="Frame"
    )
)

interactive(children=(IntSlider(value=0, description='Frame', max=12599), Output()), _dom_classes=('widget-int…

<function __main__.show_frame(frame_idx)>

In [26]:
from matplotlib.colors import hsv_to_rgb
out_video_flow = np.load(optic_flow_path, mmap_mode="r")

def flow_to_rgb(flow_y, flow_x, max_magnitude=None):
    mag = np.sqrt(flow_x**2 + flow_y**2)
    ang = np.arctan2(flow_y, flow_x)

    if max_magnitude is None:
        max_magnitude = np.nanpercentile(mag, 99)

    if not np.isfinite(max_magnitude) or max_magnitude <= 0:
        max_magnitude = 1.0

    norm = np.clip(mag / max_magnitude, 0, 1)

    hsv = np.zeros((*flow_y.shape, 3), dtype=np.float32)
    hsv[..., 0] = (ang + np.pi) / (2 * np.pi)
    hsv[..., 1] = norm
    hsv[..., 2] = norm

    return (hsv_to_rgb(hsv) * 255).astype(np.uint8)



In [27]:
#optic flow display
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


# Expected shapes:
# out_video:      (n_frames, width, height)
# out_video_flow: (n_frames, 4, width, height)
#
# channels:
# 0 = flow_x
# 1 = flow_y
# 2 = divergence
# 3 = energy


def inspect_stored_flow(
    frame_idx,
    show_quiver=True,
    flow_vmax=3,
    div_vmax=0.5,
    energy_vmax=None,
):
    frame_idx = int(frame_idx)

    gray = out_video[frame_idx]
    prev_gray = out_video[max(frame_idx - 1, 0)]

    flow_x = out_video_flow[frame_idx, 0]
    flow_y = out_video_flow[frame_idx, 1]
    divergence = out_video_flow[frame_idx, 2]
    energy = out_video_flow[frame_idx, 3]

    velocity_rgb = flow_to_rgb(flow_y.T, flow_x.T, max_magnitude=90)

    fig, ax = plt.subplots(1, 4, figsize=(18, 4))

    ax[0].imshow((2 * gray + prev_gray).T, cmap="jet", origin="lower")
    ax[0].set_title(f"Input frames {frame_idx-1} → {frame_idx}")

    ax[1].imshow(velocity_rgb, origin="lower")
    ax[1].set_title(
        f"Flow\n"
        f"x [{np.nanmin(flow_x):.2f}, {np.nanmax(flow_x):.2f}], "
        f"y [{np.nanmin(flow_y):.2f}, {np.nanmax(flow_y):.2f}]"
    )

    if show_quiver:
        step = max(1, min(gray.shape) // 30)

        x_idx = np.arange(step // 2, gray.shape[0], step)
        y_idx = np.arange(step // 2, gray.shape[1], step)
        xx, yy = np.meshgrid(x_idx, y_idx)

        ax[1].quiver(
            xx,
            yy,
            flow_x[xx, yy],
            flow_y[xx, yy],
            angles="xy",
            scale_units="xy",
            scale=flow_vmax,
            width=0.002,
        )

    ax[2].imshow(
        divergence.T,
        cmap="seismic",
        origin="lower",
        vmin=-div_vmax,
        vmax=div_vmax,
    )
    ax[2].set_title(
        f"Divergence\n"
        f"[{np.nanmin(divergence):.2f}, {np.nanmax(divergence):.2f}] 1/s"
    )

    if energy_vmax is None:
        energy_vmax = np.nanpercentile(energy, 99)
        if not np.isfinite(energy_vmax) or energy_vmax <= 0:
            energy_vmax = 1.0

    ax[3].imshow(
        energy.T,
        cmap="inferno",
        origin="lower",
        vmin=0,
        vmax=energy_vmax,
    )
    ax[3].set_title(
        f"Energy\n"
        f"[0, {np.nanmax(energy):.2f}] 1/s²"
    )

    for a in ax:
        a.axis("off")

    plt.tight_layout()
    plt.show()


max_frame = min(out_video.shape[0], out_video_flow.shape[0]) - 1

widgets.interact(
    inspect_stored_flow,
    frame_idx=widgets.IntSlider(
        value=1,
        min=1,
        max=max_frame,
        step=1,
        description="Frame",
        continuous_update=False,
    ),
    show_quiver=widgets.Checkbox(
        value=True,
        description="Quiver",
    ),
    flow_vmax=widgets.FloatSlider(
        value=30,
        min=0.1,
        max=200,
        description="Flow vmax",
    ),
    div_vmax=widgets.FloatSlider(
        value=30,
        min=0.1,
        max=150,
        description="Divergence vmax",
    ),
    energy_vmax=widgets.FloatSlider(
        value=50,
        description="Energy vmax",
    ),
)

interactive(children=(IntSlider(value=1, continuous_update=False, description='Frame', max=12599, min=1), Chec…

<function __main__.inspect_stored_flow(frame_idx, show_quiver=True, flow_vmax=3, div_vmax=0.5, energy_vmax=None)>

## Decomposition

In [28]:
if 'WT' in globals():
    del WT # unlink the memmapped variable as we might want to regenerate the file
    
dwt_path=compute_and_save_dwt_vS(downsampled_video_path, wavelet_params,   device='cuda', force=False)

Loaded downsampled video data from D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_scaled100x75.npy with shape (12600, 100, 75) and dtype bool
||Constructing: zebra_s0_d420.0_fps59.94_RESAMPLED30fps_scaled100x75_libvS_16_12_8_5_5_7_2_362db4c1dwt.npy 
Wavelet transform file D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_scaled100x75_libvS_16_12_8_5_5_7_2_362db4c1dwt.npy already exists. Skipping computation.


## Load and display decomposition

In [29]:
WT=np.load(dwt_path, mmap_mode='r')
print(f"Loaded DWT from {dwt_path} with shape {WT.shape}")

downsampled_video=np.load(downsampled_video_path)

Loaded DWT from D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_scaled100x75_libvS_16_12_8_5_5_7_2_362db4c1dwt.npy with shape (12600, 16, 12, 8, 5, 5, 7, 2)


In [30]:
# interactive visualization of the DWT and video frames
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

az_left, az_right, el_bottom, el_top = visual_coverage

param_names = ["x", "y", "angle", "size", "freq", "drift", "phase"]
param_values = [xs, ys, angles, sizes, freqs, drifts, phases]

if WT.shape[0] != downsampled_video.shape[0]:
    print(f"Warning: WT has {WT.shape[0]} frames but video has {downsampled_video.shape[0]} frames!")
n_frames = downsampled_video.shape[0]

sliders = [
    widgets.IntSlider(
        value=len(vals) // 2,
        min=0,
        max=len(vals) - 1,
        step=1,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="250px")
    )
    for name, vals in zip(param_names, param_values)
]

time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=n_frames - 1,
    step=1,
    description="frame",
    continuous_update=False,
    layout=widgets.Layout(width="250px")
)

out_zoom = widgets.Output()
out_full = widgets.Output()
out_overlay = widgets.Output()
out_xy = widgets.Output()

def plot_view(*args):
    xi, yi, anglei, sizei, freqi, drifti, phasei = [s.value for s in sliders]
    ti = time_slider.value

    frame = downsampled_video[ti]
    filt = get_filter_from_params(xi, yi, anglei, sizei, freqi, drifti, phasei, wavelet_params)
    xv, yv, dx, dy = get_filter_vector(filt, visual_coverage)


    transient = WT[:, xi, yi, anglei, sizei, freqi, drifti,  phasei]

    # x-y plane at current time and selected non-spatial parameters
    xy_plane = WT[ti, :, :, anglei, sizei, freqi, drifti,  phasei]

    title_params = (
        f"frame={ti}, "
        f"x={xs[xi]:.2f}, y={ys[yi]:.2f}, "
        f"angle={angles[anglei]:.2f}, "
        f"size={sizes[sizei]:.2f}, "
        f"freq={freqs[freqi]:.3f}, "
        f"drift={drifts[drifti]:.2f}, "
        f"phase={phases[phasei]:.2f}"
    )

    # --- TOP RIGHT: temporal zoom plot
    with out_zoom:
        out_zoom.clear_output(wait=True)

        z0 = max(0, ti - 12)
        z1 = min(n_frames, ti + 13)

        fig, ax = plt.subplots(1,1,figsize=(6, 3))
        ax.plot(np.arange(z0, z1), transient[z0:z1], marker="o")
        ax.axvline(ti, color="red", linestyle="--", linewidth=2)
        ax.set_xlim(z0, z1 - 1)
        ax.set_title("WT transient — zoom ±12 frames")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Response")
        ax.set_ylim(transient.min() , transient.max() )
        plt.tight_layout()
        plt.show()

    # --- MIDDLE: full temporal plot
    with out_full:
        out_full.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(transient)
        ax.axvline(ti, color="red", linestyle="--", linewidth=2)
        ax.set_xlim(0, n_frames - 1)
        ax.set_title("WT transient — full")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Response")
        plt.tight_layout()
        plt.show()

    # --- BOTTOM LEFT: current video frame with Gabor overlay
    with out_overlay:
        out_overlay.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(6, 4))

        ax.imshow(
            frame.T,
            cmap="gray",
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )

        filt_plot = filt[filt.shape[0] // 2].T
        v = np.max(np.abs(filt_plot))
        if v == 0:
            v = 1

        rgba = np.zeros((*filt_plot.shape, 4), dtype=float)
        pos = filt_plot > 0
        neg = filt_plot < 0

        rgba[pos, 0] = 1.0   # red = positive
        rgba[neg, 1] = 1.0   # green = negative
        rgba[..., 3] = np.abs(filt_plot) / v * 0.7

        ax.imshow(
            rgba,
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )
        
        ax.arrow(
            xv, yv,   # start
            dx, dy,         # vector
            head_width=1,
            length_includes_head=True, 
            color="blue"
        )

        ax.set_title("Video + selected Gabor overlay")
        ax.set_xlabel("Azimuth (°)")
        ax.set_ylabel("Elevation (°)")
        plt.tight_layout()
        plt.show()

    # --- BOTTOM RIGHT: x-y WT plane
    with out_xy:
        out_xy.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(6, 4))

        im = ax.imshow(
            xy_plane.T,
            cmap="seismic",
            vmin=-0.4,
            vmax=0.4,
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )

        ax.arrow(
            xv, yv,   # start
            dx, dy,         # vector
            head_width=1,
            length_includes_head=True, 
            color="blue"
        )
        
        ax.scatter(xs[xi], ys[yi], c="yellow", s=60, edgecolors="black")
        ax.set_title("WT x-y plane at selected frame")
        ax.set_xlabel("Azimuth (°)")
        ax.set_ylabel("Elevation (°)")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        plt.show()

    sliders[0].description = "x"
    sliders[1].description = "y"

# update on slider movement
for s in sliders + [time_slider]:
    s.observe(plot_view, names="value")

slider_box = widgets.VBox([time_slider] + sliders)

top_row = widgets.HBox([slider_box, out_zoom])
bottom_row = widgets.HBox([out_overlay, out_xy])

display(top_row, out_full, bottom_row)

plot_view()

Output()